# 03 — Gold Aggregations

Purpose: create a business-ready Gold table from clean Silver events.

Process schema:

```text
SILVER EVENTS
+----------+---------+------------+--------+---------------------+
| event_id | user_id | event_type | amount | event_time          |
+----------+---------+------------+--------+---------------------+
| e1       | u1      | purchase   | 10.0   | 2026-01-01 10:00:00 |
| e2       | u1      | purchase   | 15.0   | 2026-01-01 10:05:00 |
| e3       | u1      | refund     | 5.0    | 2026-01-01 10:10:00 |
| e4       | u2      | purchase   | 20.0   | 2026-01-01 11:00:00 |
| e5       | u2      | purchase   | 30.0   | 2026-01-02 09:00:00 |
+----------+---------+------------+--------+---------------------+

                       GROUP BY user_id, event_date
                                  |
                                  v

GOLD USER DAILY METRICS
+---------+------------+-------------+----------------+-------------+---------------+-----------+
| user_id | event_date | event_count | purchase_count | refund_count| gross_sales   | net_sales |
+---------+------------+-------------+----------------+-------------+---------------+-----------+
| u1      | 2026-01-01 | 3           | 2              | 1           | 25.0          | 20.0      |
| u2      | 2026-01-01 | 1           | 1              | 0           | 20.0          | 20.0      |
| u2      | 2026-01-02 | 1           | 1              | 0           | 30.0          | 30.0      |
+---------+------------+-------------+----------------+-------------+---------------+-----------+
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-gold-aggregations")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Silver events

Input is already clean Silver data.

Gold does not fix raw quality problems. Gold creates business-level results.

In [2]:
silver_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("event_time_raw", StringType(), False),
])

rows = [
    ("e1", "u1", "purchase", 10.0, "2026-01-01 10:00:00"),
    ("e2", "u1", "purchase", 15.0, "2026-01-01 10:05:00"),
    ("e3", "u1", "refund", 5.0, "2026-01-01 10:10:00"),
    ("e4", "u2", "purchase", 20.0, "2026-01-01 11:00:00"),
    ("e5", "u2", "purchase", 30.0, "2026-01-02 09:00:00"),
    ("e6", "u3", "refund", 8.0, "2026-01-02 12:00:00"),
]

silver_events = (
    spark.createDataFrame(rows, silver_schema)
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .drop("event_time_raw")
)

silver_events.show(truncate=False)
silver_events.printSchema()

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time         |
+--------+-------+----------+------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|
|e2      |u1     |purchase  |15.0  |2026-01-01 10:05:00|
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|
|e5      |u2     |purchase  |30.0  |2026-01-02 09:00:00|
|e6      |u3     |refund    |8.0   |2026-01-02 12:00:00|
+--------+-------+----------+------+-------------------+

root
 |-- event_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- event_type: string (nullable = false)
 |-- amount: double (nullable = false)
 |-- event_time: timestamp (nullable = true)



## Step 2 — Add reporting grain

Gold tables should have a clear grain.

```text
grain = one row per user_id per event_date
```

In [3]:
events_with_date = silver_events.withColumn("event_date", F.to_date("event_time"))

events_with_date.show(truncate=False)

+--------+-------+----------+------+-------------------+----------+
|event_id|user_id|event_type|amount|event_time         |event_date|
+--------+-------+----------+------+-------------------+----------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01|
|e2      |u1     |purchase  |15.0  |2026-01-01 10:05:00|2026-01-01|
|e3      |u1     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01|
|e4      |u2     |purchase  |20.0  |2026-01-01 11:00:00|2026-01-01|
|e5      |u2     |purchase  |30.0  |2026-01-02 09:00:00|2026-01-02|
|e6      |u3     |refund    |8.0   |2026-01-02 12:00:00|2026-01-02|
+--------+-------+----------+------+-------------------+----------+



## Step 3 — Build Gold metrics

Aggregation rules:

```text
event_count    = all events
purchase_count = count purchase events
refund_count   = count refund events
gross_sales    = sum purchase amount
refund_amount  = sum refund amount
net_sales      = gross_sales - refund_amount
```

In [4]:
gold_user_daily = (
    events_with_date
    .groupBy("user_id", "event_date")
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.when(F.col("event_type") == "purchase", F.lit(1)).otherwise(F.lit(0))).alias("purchase_count"),
        F.sum(F.when(F.col("event_type") == "refund", F.lit(1)).otherwise(F.lit(0))).alias("refund_count"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("amount")).otherwise(F.lit(0.0))).alias("gross_sales"),
        F.sum(F.when(F.col("event_type") == "refund", F.col("amount")).otherwise(F.lit(0.0))).alias("refund_amount"),
    )
    .withColumn("net_sales", F.col("gross_sales") - F.col("refund_amount"))
    .orderBy("user_id", "event_date")
)

gold_user_daily.show(truncate=False)
gold_user_daily.printSchema()

[Stage 4:>                                                          (0 + 2) / 2]

+-------+----------+-----------+--------------+------------+-----------+-------------+---------+
|user_id|event_date|event_count|purchase_count|refund_count|gross_sales|refund_amount|net_sales|
+-------+----------+-----------+--------------+------------+-----------+-------------+---------+
|u1     |2026-01-01|3          |2             |1           |25.0       |5.0          |20.0     |
|u2     |2026-01-01|1          |1             |0           |20.0       |0.0          |20.0     |
|u2     |2026-01-02|1          |1             |0           |30.0       |0.0          |30.0     |
|u3     |2026-01-02|1          |0             |1           |0.0        |8.0          |-8.0     |
+-------+----------+-----------+--------------+------------+-----------+-------------+---------+

root
 |-- user_id: string (nullable = false)
 |-- event_date: date (nullable = true)
 |-- event_count: long (nullable = false)
 |-- purchase_count: long (nullable = true)
 |-- refund_count: long (nullable = true)
 |-- gross

## Step 4 — Optional business filter

Example:

```text
active sales days = days with gross_sales > 0
```

In [5]:
gold_active_sales_days = gold_user_daily.filter(F.col("gross_sales") > 0)

gold_active_sales_days.show(truncate=False)

+-------+----------+-----------+--------------+------------+-----------+-------------+---------+
|user_id|event_date|event_count|purchase_count|refund_count|gross_sales|refund_amount|net_sales|
+-------+----------+-----------+--------------+------------+-----------+-------------+---------+
|u1     |2026-01-01|3          |2             |1           |25.0       |5.0          |20.0     |
|u2     |2026-01-01|1          |1             |0           |20.0       |0.0          |20.0     |
|u2     |2026-01-02|1          |1             |0           |30.0       |0.0          |30.0     |
+-------+----------+-----------+--------------+------------+-----------+-------------+---------+



## Step 5 — Control totals

Control totals compare Silver source amounts with Gold aggregated amounts.

The `value` column is explicitly `DoubleType`, because counts are integers and amounts are doubles.

In [6]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", DoubleType(), True),
])

control_rows = [
    ("silver_event_rows", float(silver_events.count())),
    ("gold_user_daily_rows", float(gold_user_daily.count())),
    (
        "silver_total_purchase_amount",
        float(silver_events.filter("event_type = 'purchase'").agg(F.sum("amount")).first()[0] or 0.0),
    ),
    (
        "gold_total_gross_sales",
        float(gold_user_daily.agg(F.sum("gross_sales")).first()[0] or 0.0),
    ),
    (
        "silver_total_refund_amount",
        float(silver_events.filter("event_type = 'refund'").agg(F.sum("amount")).first()[0] or 0.0),
    ),
    (
        "gold_total_refund_amount",
        float(gold_user_daily.agg(F.sum("refund_amount")).first()[0] or 0.0),
    ),
]

control_totals = spark.createDataFrame(control_rows, control_totals_schema)

control_totals.show(truncate=False)

+----------------------------+-----+
|metric                      |value|
+----------------------------+-----+
|silver_event_rows           |6.0  |
|gold_user_daily_rows        |4.0  |
|silver_total_purchase_amount|75.0 |
|gold_total_gross_sales      |75.0 |
|silver_total_refund_amount  |13.0 |
|gold_total_refund_amount    |13.0 |
+----------------------------+-----+



## Final result

```text
SILVER EVENTS
      |
      v
ADD REPORTING DATE
      |
      v
GROUP BY user_id, event_date
      |
      v
GOLD USER DAILY METRICS
```

Gold output is no longer raw events. It is a reporting/business table with a defined grain.

In [7]:
spark.stop()